Original data retrieved 24/08/02 from: 
    https://cloud.ilabt.imec.be/index.php/s/6zr8GcEBgJQqqzj/download/plantclef.zip

see github:
    https://github.com/kymillev/herbarium-segmentation

This is the Code used to convert plantclef (Milleville et al. 2023) dataset to a semantic segmentation set. Note in their work, they report validation results which were used to optimize training. To this end, we prefer to call their validation data testing data and randomly partition out a new validation set.

Post harmonization, the classes should be:

Dataset alignment should result in the Milleville class ID changes:
- 4, 5, 6 => 3 # combine barcode, stamp, and attachment to notes class
- 8 => 0 # convert other to 0 to merge with "background"

Dataset alignment should result in the synthetic class ID changes:
- 5 => 0 # convert paper to other background class
- 150, 250 => 1 # bark and leaf to new plant class
- 100 => 2 # scale to new scalebar class
- 200 => 3 # label to new notes class
- 50 => 4 # crc to new crc class

Final results should have a network with these 5 classes:
- 0: background/other/paper
- 1: plant
- 2: scalebar
- 3: notes
- 4: crc

In [1]:
#Synthetic class IDs:
syn_class_ids = {'other':0,
             'paper':5,
             'crc':50,
             'scale':100,
             'bark':150,
             'label':200,
             'leaf':250}

#plantclef IDs:
pc_ids = {'plant':1, 
          'ruler':2,
          'note':3,
          'barcode':4,
          'stamp':5,
          'attachment':6,
          'color_card':7,
          'other':8}

In [2]:
# After installing "pycocotools" and "panopticapi" 
# see: "https://github.com/cocodataset/panopticapi" for panopticapi installation

# this script is from "https://github.com/cocodataset/panopticapi/blob/master/converters/panoptic2semantic_segmentation.py"
#  has been modified for notebook use

'''
This script converts data in panoptic COCO format to semantic segmentation. All
segments with the same semantic class in one image are combined together.

Additional option:
- using option '--things_others' the script combine all segments of thing
classes into one segment with semantic class 'other'.
'''
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
from __future__ import unicode_literals
import os, sys
import argparse
import numpy as np
import json
import time
import multiprocessing
from collections import defaultdict

import PIL.Image as Image

from panopticapi.utils import get_traceback, rgb2id, save_json

try:
    # set up path for pycocotools
    # sys.path.append('./cocoapi-master/PythonAPI/')
    from pycocotools import mask as COCOmask
except Exception:
    raise Exception("Please install pycocotools module from https://github.com/cocodataset/cocoapi")

OTHER_CLASS_ID = 183

@get_traceback
def extract_semantic_single_core(proc_id,
                                 annotations_set,
                                 segmentations_folder,
                                 output_json_file,
                                 semantic_seg_folder,
                                 categories,
                                 save_as_png,
                                 things_other):
    annotation_semantic_seg = []
    for working_idx, annotation in enumerate(annotations_set):
        if working_idx % 100 == 0:
            print('Core: {}, {} from {} images processed'.format(proc_id,
                                                                 working_idx,
                                                                 len(annotations_set)))
        try:
            pan_format = np.array(
                Image.open(os.path.join(segmentations_folder, annotation['file_name'])),
                dtype=np.uint32
            )
        except IOError:
            raise KeyError('no prediction png file for id: {}'.format(annotation['image_id']))

        pan = rgb2id(pan_format)
        semantic = np.zeros(pan.shape, dtype=np.uint8)

        RLE_per_category = defaultdict(list)
        for segm_info in annotation['segments_info']:
            cat_id = segm_info['category_id']
            if things_other and categories[cat_id]['isthing'] == 1:
                cat_id = OTHER_CLASS_ID
            mask = pan == segm_info['id']
            if save_as_png:
                semantic[mask] = cat_id
            else:
                RLE = COCOmask.encode(np.asfortranarray(mask.astype('uint8')))
                RLE['counts'] = RLE['counts'].decode('utf8')
                RLE_per_category[cat_id].append(RLE)

        if save_as_png:
            Image.fromarray(semantic).save(os.path.join(semantic_seg_folder, annotation['file_name']))
        else:
            for cat_id, RLE_list in RLE_per_category.items():
                if len(RLE_list) == 1:
                    RLE = RLE_list[0]
                else:
                    RLE = COCOmask.merge(RLE_list)
                semantic_seg_record = {}
                semantic_seg_record["image_id"] = annotation['image_id']
                semantic_seg_record["category_id"] = cat_id
                semantic_seg_record["segmentation"] = RLE
                semantic_seg_record["area"] = int(COCOmask.area(RLE))
                semantic_seg_record["bbox"] = list(COCOmask.toBbox(RLE))
                semantic_seg_record["iscrowd"] = 0
                annotation_semantic_seg.append(semantic_seg_record)
    print('Core: {}, all {} images processed'.format(proc_id, len(annotations_set)))

    return annotation_semantic_seg


def extract_semantic(input_json_file,
                     segmentations_folder,
                     output_json_file,
                     semantic_seg_folder,
                     categories_json_file,
                     things_other):
    start_time = time.time()
    with open(input_json_file, 'r') as f:
        d_coco = json.load(f)
    annotations = d_coco['annotations']

    if segmentations_folder is None:
        segmentations_folder = input_json_file.rsplit('.', 1)[0]

    print("EXTRACTING FROM...")
    print("COCO panoptic format:")
    print("\tSegmentation folder: {}".format(segmentations_folder))
    print("\tJSON file: {}".format(input_json_file))
    print("SEMANTIC SEGMENTATION")

    if output_json_file is not None and semantic_seg_folder is not None:
        raise Exception("'--output_json_file' and '--semantic_seg_folder' \
                        options cannot be used together")

    save_as_png = False
    if output_json_file is None:
        if semantic_seg_folder is None:
            raise Exception("One of '--output_json_file' and '--semantic_seg_folder' \
                            options must be used specified")
        else:
            save_as_png = True
            print("in PNG format:")
            print("\tFolder with semnatic segmentations: {}".format(semantic_seg_folder))
            if not os.path.isdir(semantic_seg_folder):
                print("Creating folder {} for semantic segmentation PNGs".format(semantic_seg_folder))
                os.mkdir(semantic_seg_folder)
    else:
        print("in COCO detection format:")
        print("\tJSON file: {}".format(output_json_file))
    if things_other:
        print("Merging all things categories into 'other' category")
    print('\n')

    with open(categories_json_file, 'r') as f:
        categories_list = json.load(f)
    categories = {category['id']: category for category in categories_list}

    cpu_num = multiprocessing.cpu_count()
    annotations_split = np.array_split(annotations, cpu_num)
    print("Number of cores: {}, images per core: {}".format(cpu_num, len(annotations_split[0])))
    workers = multiprocessing.Pool(processes=cpu_num)
    processes = []
    for proc_id, annotations_set in enumerate(annotations_split):
        p = workers.apply_async(extract_semantic_single_core,
                                (proc_id, annotations_set, segmentations_folder,
                                 output_json_file, semantic_seg_folder,
                                 categories, save_as_png, things_other))
        processes.append(p)
    annotations_coco_semantic_seg = []
    for p in processes:
        annotations_coco_semantic_seg.extend(p.get())

    if not save_as_png:
        for idx, ann in enumerate(annotations_coco_semantic_seg):
            ann['id'] = idx
        d_coco['annotations'] = annotations_coco_semantic_seg
        categories_coco_semantic_seg = []
        for category in categories_list:
            if things_other and category['isthing'] == 1:
                continue
            category.pop('isthing')
            category.pop('color')
            categories_coco_semantic_seg.append(category)
        if things_other:
            categories_coco_semantic_seg.append({'id': OTHER_CLASS_ID,
                                                 'name': 'other',
                                                 'supercategory': 'other'})
        d_coco['categories'] = categories_coco_semantic_seg
        save_json(d_coco, output_json_file)

    t_delta = time.time() - start_time
    print("Time elapsed: {:0.2f} seconds".format(t_delta))


# if __name__ == "__main__":
#     parser = argparse.ArgumentParser(
#         description="This script converts data in panoptic COCO format to \
#         semantic segmentation. All segments with the same semantic class in one \
#         image are combined together. See this file's head for more information."
#     )
#     parser.add_argument('--input_json_file', type=str,
#                         help="JSON file with panoptic data")
#     parser.add_argument(
#         '--segmentations_folder', type=str, default=None, help="Folder with \
#          panoptic COCO format segmentations. Default: X if input_json_file is \
#          X.json"
#     )
#     parser.add_argument('--output_json_file', type=str, default=None,
#                         help="JSON file with semantic data. If '--output_json_file' \
#                         is specified, resulting semantic segmentation will be \
#                         stored as a JSON file in COCO stuff format (see \
#                         http://cocodataset.org/#format-data for details).")
#     parser.add_argument('--semantic_seg_folder', type=str, default=None,
#                         help="Folder for semantic segmentation. If '--semantic_seg_folder' \
#                         is specified, resulting semantic segmentation will be \
#                         stored in the specified folder in PNG format.")
#     parser.add_argument('--categories_json_file', type=str,
#                         help="JSON file with Panoptic COCO categories information",
#                         default='./panoptic_coco_categories.json')
#     parser.add_argument('--things_other', action='store_true',
#                         help="Is set, all things classes are merged into one \
#                         'other' class")
#     args = parser.parse_args()
#     extract_semantic(args.input_json_file,
#                      args.segmentations_folder,
#                      args.output_json_file,
#                      args.semantic_seg_folder,
#                      args.categories_json_file,
#                      args.things_other)

## Once run, this was commented out to avoid rerunning.

# # Convert the validation data
# # Note, the authors report data from a validation set used to optimize training.
# # We prefer to consider this a testing set, hence porting the output to the plantclef_test_masks subdirectory.
# input_json_file = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/panoptic_val.json"
# segmentations_folder = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/panoptic_val/"
# output_json_file = None
# semantic_seg_folder = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_test_masks/"
# categories_json_file = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/panoptic_herbaria_categories.json"
# things_other = False
# extract_semantic(input_json_file,
#                      segmentations_folder,
#                      output_json_file,
#                      semantic_seg_folder,
#                      categories_json_file,
#                      things_other)

# # Convert the train data
# input_json_file = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/panoptic_train.json"
# segmentations_folder = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/panoptic_train/"
# output_json_file = None
# semantic_seg_folder = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_train_masks/"
# categories_json_file = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/panoptic_herbaria_categories.json"
# things_other = False
# extract_semantic(input_json_file,
#                      segmentations_folder,
#                      output_json_file,
#                      semantic_seg_folder,
#                      categories_json_file,
#                      things_other)

In [3]:
import os
import random
import shutil

# Define paths
image_dir = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_train_images"
mask_dir = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_train_masks"
val_image_dir = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_val_images"
val_mask_dir = "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_val_masks"

# Create validation directories if they don't exist
os.makedirs(val_image_dir, exist_ok=True)
os.makedirs(val_mask_dir, exist_ok=True)

# List all image files in the image directory
image_files = [f for f in os.listdir(image_dir) if f.endswith('.jpg')]

# Randomly select a subset of image files
num_images_to_select = 30
selected_images = random.sample(image_files, num_images_to_select)
selected_images

['102155.jpg',
 '248974.jpg',
 '13139.jpg',
 '234123.jpg',
 '142447.jpg',
 'E00058993.jpg',
 '230155.jpg',
 '257878.jpg',
 'BM000551636.jpg',
 '249449.jpg',
 '112682.jpg',
 '12858.jpg',
 '196181.jpg',
 '256769.jpg',
 '121432.jpg',
 '175126.jpg',
 '246264.jpg',
 'EIG.3022.jpg',
 '135863.jpg',
 '262748.jpg',
 '200107.jpg',
 'EIG.1060.jpg',
 '230258.jpg',
 '20683.jpg',
 '11125.jpg',
 '10321.jpg',
 '130309.jpg',
 '146685.jpg',
 '112949.jpg',
 '212518.jpg']

In [4]:
# Move selected images and corresponding masks to the validation directories
for image_file in selected_images:
    # Define full paths
    image_path = os.path.join(image_dir, image_file)
    mask_path = os.path.join(mask_dir, image_file.replace('.jpg', '.png'))
    
    # Define validation paths
    val_image_path = os.path.join(val_image_dir, image_file)
    val_mask_path = os.path.join(val_mask_dir, image_file.replace('.jpg', '.png'))
    
    # Move files
    shutil.move(image_path, val_image_path)
    shutil.move(mask_path, val_mask_path)

print(f"Moved {len(selected_images)} images and their masks to validation directories.")


Moved 30 images and their masks to validation directories.


Now, these files need an identifier appended to their name for the run_experiments.py to work.

In [5]:
import os

# Define directories for training, testing, and validation
directories = {
    "train_images": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_train_images",
    "train_masks": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_train_masks",
    "val_images": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_val_images",
    "val_masks": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_val_masks",
    "test_images": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_test_images",
    "test_masks": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_test_masks"
}

# Define suffixes for images and masks
suffixes = {
    "images": "_src",
    "masks": "_msk"
}

# Function to rename files in a directory
def rename_files(directory, suffix):
    for filename in os.listdir(directory):
        # Split the filename and extension
        base, ext = os.path.splitext(filename)
        
        # Create new filename
        new_filename = f"{base}{suffix}{ext}"
        
        # Define full paths
        old_path = os.path.join(directory, filename)
        new_path = os.path.join(directory, new_filename)
        
        # Rename file
        os.rename(old_path, new_path)

        print(f"Renamed '{filename}' to '{new_filename}'")

# Rename files in each directory
for key, directory in directories.items():
    if "images" in key:
        rename_files(directory, suffixes["images"])
    elif "masks" in key:
        rename_files(directory, suffixes["masks"])

Renamed 'K000996455.jpg' to 'K000996455_src.jpg'
Renamed 'K000930539.jpg' to 'K000930539_src.jpg'
Renamed 'EIG.7613.jpg' to 'EIG.7613_src.jpg'
Renamed 'EIG.2382.jpg' to 'EIG.2382_src.jpg'
Renamed 'EIG.2379.jpg' to 'EIG.2379_src.jpg'
Renamed 'E00134537.jpg' to 'E00134537_src.jpg'
Renamed 'E00067488.jpg' to 'E00067488_src.jpg'
Renamed 'E00065427.jpg' to 'E00065427_src.jpg'
Renamed 'E00016304.jpg' to 'E00016304_src.jpg'
Renamed 'BM000798777.jpg' to 'BM000798777_src.jpg'
Renamed 'BM000625363.jpg' to 'BM000625363_src.jpg'
Renamed 'BM000625345.jpg' to 'BM000625345_src.jpg'
Renamed 'BM000521866.jpg' to 'BM000521866_src.jpg'
Renamed 'BM000521755.jpg' to 'BM000521755_src.jpg'
Renamed 'B_10_0157339.jpg' to 'B_10_0157339_src.jpg'
Renamed 'B_10_0086706.jpg' to 'B_10_0086706_src.jpg'
Renamed 'B_10_0003487.jpg' to 'B_10_0003487_src.jpg'
Renamed '277955.jpg' to '277955_src.jpg'
Renamed '274366.jpg' to '274366_src.jpg'
Renamed '272535.jpg' to '272535_src.jpg'
Renamed '270005.jpg' to '270005_src.jpg'
R

Next up, for ease of processing all images will be rotated 90 degrees into landscape. This simply aligns with the synthetic data and other experiments making preprocessing simpler. 

In [6]:
import os
from PIL import Image

# Define directories for training, testing, and validation
directories = {
    "train_images": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_train_images",
    "train_masks": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_train_masks",
    "val_images": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_val_images",
    "val_masks": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_val_masks",
    "test_images": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_test_images",
    "test_masks": "/run/media/john/Storage/synthetic_segmentation/dataset/plantclef/converted/plantclef_test_masks"
}

# Function to rotate images
def rotate_images(directory):
    for filename in os.listdir(directory):
        if filename.endswith(('.jpg', '.png')):  # Check for image file extensions
            # Full path to image
            image_path = os.path.join(directory, filename)
            
            # Open image
            with Image.open(image_path) as img:
                # Rotate image 90 degrees clockwise
                rotated_img = img.rotate(-90, expand=True)  # -90 for clockwise
                
                # Save rotated image
                rotated_img.save(image_path)

            print(f"Rotated '{filename}'")

# Apply rotation to each directory
for key, directory in directories.items():
    rotate_images(directory)


Rotated 'EIG.7613_src.jpg'
Rotated 'EIG.2382_src.jpg'
Rotated 'EIG.2379_src.jpg'
Rotated 'E00134537_src.jpg'
Rotated 'E00067488_src.jpg'
Rotated 'E00065427_src.jpg'
Rotated 'E00016304_src.jpg'
Rotated 'BM000798777_src.jpg'
Rotated 'BM000625363_src.jpg'
Rotated 'BM000625345_src.jpg'
Rotated 'BM000521866_src.jpg'
Rotated 'BM000521755_src.jpg'
Rotated 'B_10_0157339_src.jpg'
Rotated 'B_10_0086706_src.jpg'
Rotated 'B_10_0003487_src.jpg'
Rotated '277955_src.jpg'
Rotated '274366_src.jpg'
Rotated '272535_src.jpg'
Rotated '270005_src.jpg'
Rotated '269506_src.jpg'
Rotated '269465_src.jpg'
Rotated '267683_src.jpg'
Rotated '266925_src.jpg'
Rotated '266670_src.jpg'
Rotated '266584_src.jpg'
Rotated '258370_src.jpg'
Rotated 'K000996455_src.jpg'
Rotated '254942_src.jpg'
Rotated '252984_src.jpg'
Rotated '249448_src.jpg'
Rotated '249123_src.jpg'
Rotated '248953_src.jpg'
Rotated '248451_src.jpg'
Rotated '245944_src.jpg'
Rotated '244782_src.jpg'
Rotated '244739_src.jpg'
Rotated '244717_src.jpg'
Rotated '2